Component reference for the completed project. The final integrated implementation, results, rubric evidence and disclosure are in `final_multi_agent_financial_analysis.ipynb`. Historical execution outputs retain their original provenance.


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys
TEAM_BRANCH = "codex/final-team-integration"  # Use main after the integration PR is merged.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    REPO = Path("/content/NLP_AAI520-team")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", TEAM_BRANCH, "--single-branch",
                        "https://github.com/tim17hays/NLP_AAI520.git", str(REPO)], check=True)
else:
    REPO = Path.cwd()
    if REPO.name == "notebooks":
        REPO = REPO.parent
if not (REPO / "src/workflow.py").exists():
    raise RuntimeError("Open this notebook from the combined repository checkout.")
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

## Install once in the GPU runtime
The setup cell uses a separate clone in Colab and preserves an existing clone. Check its printed commit; update it deliberately when retesting newer work. Locally, use your project virtual environment. Large model weights download on first live use.

In [ ]:
INSTALL_DEPENDENCIES = False  # Set True in a fresh Colab GPU runtime.
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
repo_string = str(REPO)
if repo_string not in sys.path:
    sys.path.insert(0, repo_string)
from src.planning import ResearchPlanner
from src.evaluation_memory import qwen_call_model
from src.workflow import run_two_run_experiment
print("Shared planner, evaluator, revision, memory and two-run driver imported.")

## Actual two-run workflow
Set `RUN_LIVE = True` only in a configured GPU runtime. Select a memory file that should persist; Colab runtime storage is temporary, so mount your own Drive and change this path if persistence across sessions is needed. Do not overwrite an existing memory file to manufacture a clean result.

This runs planner → retrieval/FinBERT → Qwen draft → evaluation/revision → save lesson, then retrieves memory before run 2. No fabricated fallback scores are used. A 10/10 score without a new lesson is not a successful memory demonstration.

In [ ]:
RUN_LIVE = False
MEMORY_PATH = REPO / "data/processed/research_lessons.json"
OUTPUT_DIR = REPO / "outputs"
if RUN_LIVE:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime before enabling live inference.")
    experiment = run_two_run_experiment(
        ResearchPlanner(), "AAPL",
        "Summarize market performance, financial metrics and news with source citations and limitations.",
        qwen_call_model, memory_path=MEMORY_PATH, output_dir=OUTPUT_DIR,
        execution_mode="live",
    )
    import json
    print(json.dumps({k: v for k, v in experiment.items() if k != "runs"}, indent=2))
    for number, run in enumerate(experiment["runs"], 1):
        print("RUN", number)
        print("Initial draft:", run["initial_draft"])
        print("Revised draft:", run["revised_analysis"])
        print("Plan:", json.dumps(run["plan"], indent=2))
        print("Evaluation history:", json.dumps(run["revision_history"], indent=2))
        print("Lessons received:", json.dumps(run["lessons_used"], indent=2))
        print("Saved lesson:", json.dumps(run["saved_lesson"], indent=2))
else:
    print("Live workflow not executed. Enable RUN_LIVE in a configured GPU runtime.")

Component reference for the completed project. The final integrated implementation, results, rubric evidence and disclosure are in `final_multi_agent_financial_analysis.ipynb`. Historical execution outputs retain their original provenance.
